# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'pelayosierra@protonmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
%pip install seaborn

# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

dataset = datasets.load_iris()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['species'] = dataset.target
print(f"Shape:\n{df.shape}")
print(f"\nHead:\n{df.head()}")
print(f"\nDescribe:\n{df.describe()}")


### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
# -- AVISO -- Esta sección ha sido parcialmente resuelta con el uso de IA generativa
# Mapeo de especies para leyendas legibles
species_map = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
df['species_name'] = df['species'].map(species_map)

fig, axes = plt.subplots(3, 2, figsize=(15, 18))
fig.suptitle('Exploración del Dataset Iris', fontsize=16, fontweight='bold')

# 1. Scatter: longitud vs ancho del pétalo por especie
ax = axes[0, 0]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.scatter(subset['petal length (cm)'], subset['petal width (cm)'],
               label=spec, alpha=0.7, s=60, c=color, edgecolors='white')
ax.set_xlabel('Longitud del pétalo (cm)')
ax.set_ylabel('Ancho del pétalo (cm)')
ax.set_title('Pétalo: Longitud vs Ancho')
ax.legend()

# 2. Scatter: longitud vs ancho del sépalo por especie
ax = axes[0, 1]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.scatter(subset['sepal length (cm)'], subset['sepal width (cm)'],
               label=spec, alpha=0.7, s=60, c=color, edgecolors='white')
ax.set_xlabel('Longitud del sépalo (cm)')
ax.set_ylabel('Ancho del sépalo (cm)')
ax.set_title('Sépalo: Longitud vs Ancho')
ax.legend()

# 3. Boxplots de longitud del pétalo por especie
ax = axes[1, 0]
sns.boxplot(data=df, x='species_name', y='petal length (cm)', ax=ax, palette='Set2')
ax.set_title('Distribución: Longitud del Pétalo por Especie')
ax.set_xlabel('Especie')

# 4. Histograma de longitud del pétalo por especie
ax = axes[1, 1]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.hist(subset['petal length (cm)'], bins=12, alpha=0.5, label=spec, color=color)
ax.set_xlabel('Longitud del pétalo (cm)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma: Longitud del Pétalo')
ax.legend()

# 5. Histograma de longitud del sépalo por especie
ax = axes[2, 0]
for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
    subset = df[df['species_name'] == spec]
    ax.hist(subset['sepal length (cm)'], bins=12, alpha=0.5, label=spec, color=color)
ax.set_xlabel('Longitud del sépalo (cm)')
ax.set_ylabel('Frecuencia')
ax.set_title('Histograma: Longitud del Sépalo')
ax.legend()

# 6. Matriz de correlación
ax = axes[2, 1]
corr = df.drop(columns=['species', 'species_name']).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            xticklabels=['Sep.L', 'Sep.W', 'Pet.L', 'Pet.W'],
            yticklabels=['Sep.L', 'Sep.W', 'Pet.L', 'Pet.W'])
ax.set_title('Matriz de Correlación')

plt.tight_layout()
plt.show()

print("\nObservaciones clave:")
print("   • PÉTALO: Setosa es claramente separable (pétalos cortos y estrechos <2cm). Versicolor y Virginica se distinguen bastante bien.")
print("   • SÉPALO: Mayor solapamiento entre especies, especialmente Versicolor y Virginica. Setosa tiende a tener sépalos más anchos pero cortos.")
print("   • Comparación: las medidas del pétalo separan mejor las especies que las del sépalo.")
print("   • Alta correlación entre longitud y ancho del pétalo (características redundantes).")
print("   • El sépalo tiene baja correlación entre sus dimensiones y menor poder discriminativo.")


## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# Separa características (X) y etiquetas (y).
X = df.drop(columns=['species', 'species_name'])
y = df['species']

# Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y)

# Comprueba tamaños y distribución de clases en cada conjunto
print(f"Tamaño X_train: {X_train.shape}, Tamaño X_test: {X_test.shape}")
print(f"Tamaño y_train: {y_train.shape}, Tamaño y_test: {y_test.shape}")
print(f"\nDistribución de clases - Train:\n{y_train.value_counts().sort_index()}")
print(f"\nDistribución de clases - Test:\n{y_test.value_counts().sort_index()}")
print(f"\nProporción Train (%):\n{(y_train.value_counts(normalize=True).sort_index() * 100).round(1)}")
print(f"\nProporción Test (%):\n{(y_test.value_counts(normalize=True).sort_index() * 100).round(1)}")




## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# TODO: Activa autologging de MLflow.
mlflow.sklearn.autolog()

# TODO: Inicia un run con un nombre descriptivo.
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
# Define hiperparámetros del árbol.
    max_depth = 3 # es un dataset pequeño, no requiere mucha profundidad
    max_features = None # usamos las 4
    random_state = 42
#
#   Crea y entrena DecisionTreeClassifier.
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        random_state=random_state
    )
    dt.fit(X_train, y_train)
#
#   Genera predicciones para train y test.
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
#
#   Calcula métricas de clasificación.
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average="micro")
    recall_test = recall_score(y_test, y_pred_test, average="micro")
    f1_test = 2 * (precision_test * recall_test) / (precision_test + recall_test)

    classification_metrics = {
        "accuracy_test": accuracy_test,
        "precision_test": precision_test,
        "recall_test": recall_test,
        "f1_test": f1_test
    }

    for m, val in classification_metrics.items():
        mlflow.log_metric(m, val)
#
#   Evalúa con cross-validation.
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    mlflow.log_metric("cv_accuracy", cv_scores.mean())  
    mlflow.log_metric("cv_std", cv_scores.std())
    print(cv_scores)
    print(f"CV Accuracy: {cv_scores.mean():.4f}")
    print(f"CV Std: {cv_scores.std():.4f}")
#
#   Visualiza matriz de confusión y, opcionalmente, el árbol.
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

    # Calcular matriz
    titles_options = [
        ("Confusion matrix, without normalization", None),
        ("Normalized confusion matrix", "true"),
    ]
    for title, normalize in titles_options:
        disp = ConfusionMatrixDisplay.from_estimator(
            dt,
            X_test,
            y_test,
            display_labels=["Setosa", "Versicolor", "Virginica"],
            cmap=plt.cm.Blues,
            normalize=normalize
        )
        disp.ax_.set_title(title)

        print(title)
        print(disp.confusion_matrix)

    plt.show()

    # Matriz TP / TN / FP / FN por clase (one-vs-rest)
    cm = confusion_matrix(y_test, y_pred_test)
    classes = ["Setosa", "Versicolor", "Virginica"]
    matrix_data = []
    for i in range(len(classes)):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - TP - FP - FN
        matrix_data.append([TP, TN, FP, FN])

    tp_tn_fp_fn_df = pd.DataFrame(
        matrix_data,
        index=classes,
        columns=['TP', 'TN', 'FP', 'FN']
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(tp_tn_fp_fn_df, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_title('TP / TN / FP / FN por clase (One-vs-Rest)')
    plt.tight_layout()
    plt.show()

    print("\nTP / TN / FP / FN por clase:")
    print(tp_tn_fp_fn_df)

#     TODO: Escribe una breve interpretación de los resultados.
# Los resultados demuestran que el modelo de árbol de decisión es capaz de clasificar
# correctamente la mayoría de las muestras de Iris, con una precisión del 96.7% en 
# el conjunto de prueba. Además, el modelo muestra una alta sensibilidad (recall) y 
# especificidad (precision) en todas las clases, lo que indica que es un modelo robusto 
# y generalizable. La matriz de confusión muestra que el modelo cometió un error en la 
# clasificación de una sola muestra de la clase 'Virginica'.

## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?
- ¿Qué riesgos tiene respecto a overfitting?
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

mlflow.sklearn.autolog(disable=True)
species_map = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================

configuraciones = [
    {"max_depth": 3, "max_features": None, "min_samples_split": 2},
    {"max_depth": 5, "max_features": None, "min_samples_split": 2},
    {"max_depth": 10, "max_features": None, "min_samples_split": 2},
    {"max_depth": None, "max_features": None, "min_samples_split": 2},
    {"max_depth": 3, "max_features": 2, "min_samples_split": 2},
    {"max_depth": 3, "max_features": 3, "min_samples_split": 2},
    {"max_depth": 3, "max_features": 4, "min_samples_split": 2},
    {"max_depth": 3, "max_features": 'sqrt', "min_samples_split": 2},
    {"max_depth": 5, "max_features": 2, "min_samples_split": 5},
    {"max_depth": 5, "max_features": 3, "min_samples_split": 10},
]

resultados_desafio1 = []

for i, config in enumerate(configuraciones):
    with mlflow.start_run(run_name=f"Desafío1 - Config {i+1}") as run:
        mlflow.log_params(config)

        dt = DecisionTreeClassifier(
            max_depth=config["max_depth"],
            max_features=config["max_features"],
            min_samples_split=config["min_samples_split"],
            random_state=42
        )
        dt.fit(X_train, y_train)

        y_pred = dt.predict(X_test)
        y_pred_train = dt.predict(X_train)

        acc_train = accuracy_score(y_train, y_pred_train)
        acc_test = accuracy_score(y_test, y_pred)
        cv_scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')

        mlflow.log_metric("accuracy_train", acc_train)
        mlflow.log_metric("accuracy_test", acc_test)
        mlflow.log_metric("cv_accuracy_mean", cv_scores.mean())
        mlflow.log_metric("cv_accuracy_std", cv_scores.std())

        overfit_gap = acc_train - acc_test
        mlflow.log_metric("overfit_gap", overfit_gap)

        resultados_desafio1.append({
            "config": str(config),
            "acc_train": acc_train,
            "acc_test": acc_test,
            "cv_mean": cv_scores.mean(),
            "cv_std": cv_scores.std(),
            "overfit_gap": overfit_gap
        })

        print(f"Config {i+1}: {config}")
        print(f"  Train Acc: {acc_train:.4f} | Test Acc: {acc_test:.4f} | CV: {cv_scores.mean():.4f}±{cv_scores.std():.4f} | Gap: {overfit_gap:.4f}")

df_resultados_1 = pd.DataFrame(resultados_desafio1)
print("\nResumen Desafío 1:")
print(df_resultados_1.to_string(index=False))

mejor_idx = df_resultados_1["cv_mean"].idxmax()
print(f"\nMejor configuración (por CV accuracy): {resultados_desafio1[mejor_idx]['config']}")

# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 3, 4, 'sqrt'],
    'min_samples_split': [2, 5, 10]
}

with mlflow.start_run(run_name="Desafío2 - Grid Search") as run:
    grid_search = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        return_train_score=True
    )
    grid_search.fit(X_train, y_train)

    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_accuracy", grid_search.best_score_)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy_test", acc_test)

    print(f"\nMejores parámetros: {grid_search.best_params_}")
    print(f"Mejor CV accuracy: {grid_search.best_score_:.4f}")
    print(f"Test accuracy: {acc_test:.4f}")

    cv_results = pd.DataFrame(grid_search.cv_results_)
    top5 = cv_results.nsmallest(5, 'rank_test_score')[
        ['params', 'mean_test_score', 'std_test_score', 'mean_train_score']
    ]
    print("\nTop 5 combinaciones:")
    print(top5.to_string(index=False))

# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

modelos = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42)
}

resultados = []

for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=f"Desafío3 - {nombre}") as run:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='macro')
        rec = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        cv = cross_val_score(modelo, X_train, y_train, cv=5, scoring='accuracy')

        mlflow.log_metric("accuracy_test", acc)
        mlflow.log_metric("precision_macro", prec)
        mlflow.log_metric("recall_macro", rec)
        mlflow.log_metric("f1_macro", f1)
        mlflow.log_metric("cv_accuracy_mean", cv.mean())
        mlflow.log_metric("cv_accuracy_std", cv.std())

        resultados.append({
            "Modelo": nombre,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "CV Mean": cv.mean(),
            "CV Std": cv.std()
        })

df_resultados_3 = pd.DataFrame(resultados)
print("\nComparación de modelos:")
print(df_resultados_3.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
df_resultados_3.set_index('Modelo')[['Accuracy', 'Precision', 'Recall', 'F1']].plot.bar(ax=ax)
ax.set_title('Comparación de Modelos - Iris')
ax.set_ylabel('Score')
ax.set_ylim(0.8, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# ========================================
# TODO: DESAFÍO 4 - Análisis de errores
# ========================================

y_pred_best = best_model.predict(X_test)
incorrect_indices = np.where(y_pred_best != y_test.values)[0]

print(f"\nTotal muestras mal clasificadas: {len(incorrect_indices)} / {len(y_test)}")

if len(incorrect_indices) > 0:
    errores_df = X_test.iloc[incorrect_indices].copy()
    errores_df['true_species'] = y_test.iloc[incorrect_indices].values
    errores_df['pred_species'] = y_pred_best[incorrect_indices]
    errores_df['true_name'] = errores_df['true_species'].map(species_map)
    errores_df['pred_name'] = errores_df['pred_species'].map(species_map)

    print("\nMuestras mal clasificadas:")
    print(errores_df.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    ax = axes[0]
    for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
        subset = df[df['species_name'] == spec]
        ax.scatter(subset['petal length (cm)'], subset['petal width (cm)'],
                   label=spec, alpha=0.3, s=40, c=color, edgecolors='white')
    ax.scatter(errores_df['petal length (cm)'], errores_df['petal width (cm)'],
               marker='X', s=200, c='black', label='Errores', edgecolors='red', linewidths=2)
    ax.set_xlabel('Longitud del pétalo (cm)')
    ax.set_ylabel('Ancho del pétalo (cm)')
    ax.set_title('Errores en espacio de Pétalo')
    ax.legend()

    ax = axes[1]
    for spec, color in zip(['Setosa', 'Versicolor', 'Virginica'], ['#e74c3c', '#2ecc71', '#3498db']):
        subset = df[df['species_name'] == spec]
        ax.scatter(subset['sepal length (cm)'], subset['sepal width (cm)'],
                   label=spec, alpha=0.3, s=40, c=color, edgecolors='white')
    ax.scatter(errores_df['sepal length (cm)'], errores_df['sepal width (cm)'],
               marker='X', s=200, c='black', label='Errores', edgecolors='red', linewidths=2)
    ax.set_xlabel('Longitud del sépalo (cm)')
    ax.set_ylabel('Ancho del sépalo (cm)')
    ax.set_title('Errores en espacio de Sépalo')
    ax.legend()

    plt.tight_layout()
    plt.show()

    print("\nAnálisis de confusiones:")
    for _, row in errores_df.iterrows():
        print(f"  Real: {row['true_name']} → Predicho: {row['pred_name']}")
        print(f"    Pétalo: {row['petal length (cm)']:.1f} x {row['petal width (cm)']:.1f} cm")
else:
    print("No hay muestras mal clasificadas. El modelo es perfecto en este split.")

# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================

from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"\nVarianza explicada: {pca.explained_variance_ratio_}")
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_):.4f}")

dt_2d = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_2d.fit(X_train_pca, y_train)

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
Z = dt_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 8))
cmap_bold = ListedColormap(['#e74c3c', '#2ecc71', '#3498db'])
ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_bold)

for spec, color, label in zip([0, 1, 2], ['#e74c3c', '#2ecc71', '#3498db'], ['Setosa', 'Versicolor', 'Virginica']):
    mask = y == spec
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=label, edgecolors='white', s=60, alpha=0.8)

ax.scatter(X_test_pca[:, 0], X_test_pca[:, 1], c='none', marker='o', s=120,
           edgecolors='black', linewidths=1.5, label='Test')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)')
ax.set_title('PCA 2D - Fronteras de Decisión (Decision Tree)')
ax.legend()
plt.tight_layout()
plt.show()

y_pred_pca = dt_2d.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)
print(f"\nAccuracy con 4 características: {accuracy_score(y_test, best_model.predict(X_test)):.4f}")
print(f"Accuracy con PCA (2D): {acc_pca:.4f}")
